<a id="inizio"></a>
# 06 · Pandas: Series, DataFrame e import dei dati · Soluzioni

⏱ ~90 min &nbsp;·&nbsp; Dati: `../Dati/TexasTurbine.csv`, `../Dati/letture_pod_2025.csv`, `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/utility.db`, `../Dati/fallback/lombardia_sensori.json`, `../Dati/fallback/lombardia_misure_2001.json`

**In questo notebook impariamo a**

- creare Series e DataFrame e leggere file CSV ed Excel con i parametri di lettura
- caricare una tabella da un database SQL con una query
- chiedere dati a un'API con `requests` e trasformare la risposta JSON in un DataFrame

<a id="indice"></a>
**Indice**

- [1. Importare le librerie](#sez-1)
- [2. Le strutture dati di pandas](#sez-2)
    - [2.1 Series](#sez-2-1)
    - [2.2 DataFrame](#sez-2-2)
- [3. File e cartelle con os](#sez-3)
- [4. File CSV](#sez-4)
    - [4.1 Trovare e leggere più file CSV](#sez-4-1)
    - [4.2 Specificare i tipi di dato](#sez-4-2)
    - [4.3 Un CSV salvato da Excel in italiano](#sez-4-3)
- [5. File Excel](#sez-5)
    - [5.1 Più fogli in un file](#sez-5-1)
- [6. Database SQL](#sez-6)
- [7. Lettura di dati da API](#sez-7)
    - [7.1 JSON e requests](#sez-7-1)
    - [7.2 Parametri di query](#sez-7-2)
    - [7.3 L'anagrafica dei sensori](#sez-7-3)
    - [7.4 Le misure di un sensore](#sez-7-4)
- [8. Esercizi](#sez-8)
    - [Esercizio 6.1 · Lettura di un CSV italiano](#es-6-1)
    - [Esercizio 6.2 · Due fogli Excel in una tabella](#es-6-2)
    - [Esercizio 6.3 · Le letture di un POD dal database (facoltativo)](#es-6-3)
    - [Esercizio 6.4 · La certificazione energetica degli edifici in Lombardia (facoltativo)](#es-6-4)

<a id="sez-1"></a>
## 1. Importare le librerie

Pandas è una libreria open-source per l'analisi e la manipolazione dei dati: offre strutture dati e
funzioni per lavorare con dati strutturati, come tabelle e serie temporali.

Per usare una libreria la importiamo con la parola chiave `import`. È buona pratica importare le
librerie all'inizio del notebook.

In [ ]:
import pandas as pd

<a id="sez-2"></a>
## 2. Le strutture dati di pandas

Pandas fornisce due strutture dati principali:

- **Series**: array unidimensionali etichettati;
- **DataFrame**: strutture bidimensionali con righe e colonne etichettate.

<a id="sez-2-1"></a>
### 2.1 Series

Una **Series** è un array unidimensionale che può contenere qualsiasi tipo di dato (interi, stringhe,
numeri decimali, oggetti Python). Ogni elemento è associato a un'etichetta, chiamata **indice**.

In [ ]:
# creare una Series da una lista
dati = [10, 20, 30, 40]
s = pd.Series(dati, name="valori")
s

Possiamo specificare un indice personalizzato:

In [ ]:
s_custom = pd.Series(dati, index=["a", "b", "c", "d"])
s_custom

Agli elementi di una Series si accede con l'indice, come per gli elementi di una lista.

In [ ]:
# accesso tramite indice numerico
s[0]

In [ ]:
# accesso tramite indice personalizzato
s_custom["b"]

Possiamo fare slicing, come con le liste Python:

In [ ]:
# slicing della Series
s[1:3]

<a id="sez-2-2"></a>
### 2.2 DataFrame

Un **DataFrame** è una struttura bidimensionale con dati allineati in righe e colonne, simile a un
foglio di calcolo o a una tabella SQL. Lo creiamo da un dizionario: ogni chiave diventa una colonna.

In [ ]:
# creare un DataFrame da un dizionario
dati = {
    "Nome": ["Alice", "Bob", "Charlie"],
    "Età": [24, 27, 22],
    "Città": ["Roma", "Milano", "Torino"],
}
df = pd.DataFrame(dati, index=["id_1", "id_2", "id_3"])
df

<a id="sez-3"></a>
## 3. File e cartelle con os

Il modulo `os` di Python serve a interagire con il sistema operativo: navigare tra le cartelle,
gestire file, ottenere informazioni sul sistema. `os.listdir()` elenca file e cartelle.

In [ ]:
import os

# elencare i file nella cartella di lavoro
elenco_file = os.listdir()
print("Contenuto della cartella:")
for file in elenco_file:
    print(file)

La cartella di lavoro di un notebook è quella in cui si trova il file. I dati stanno nella cartella
`Dati`, un livello più su: `..` indica la cartella superiore, quindi il percorso è `../Dati/`.

In [ ]:
os.getcwd()

In [ ]:
os.path.exists("../Dati/TexasTurbine.csv")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Stampa la lista dei file nella cartella `Dati`. Il risultato atteso è un elenco che comprende,
tra gli altri, `TexasTurbine.csv` e `utility.db`.

</div>

In [ ]:
for file in os.listdir("../Dati"):
    print(file)

<a id="sez-4"></a>
## 4. File CSV

I file CSV (*Comma-Separated Values*) sono uno dei formati più comuni per archiviare e scambiare dati
tabulari. Pandas li legge con `read_csv()`.

In [ ]:
# leggere un file CSV
df_turbina = pd.read_csv("../Dati/TexasTurbine.csv")
df_turbina.head()

`head()` mostra le prime righe. `info()` riassume righe, colonne, tipi e quanti valori non mancano;
`describe()` calcola le statistiche principali delle colonne numeriche.

In [ ]:
df_turbina.info()

In [ ]:
df_turbina.describe()

<a id="sez-4-1"></a>
### 4.1 Trovare e leggere più file CSV

Il modulo `glob` trova i file il cui nome segue uno schema, per esempio tutti quelli che finiscono
con `.csv`. Prima creiamo due file CSV di esempio.

In [ ]:
df1 = pd.DataFrame({
    "ID": [1, 2, 3],
    "Nome": ["Alice", "Bob", "Charlie"],
    "Età": ["25", "30", "35"],  # intenzionalmente come stringhe
})
df2 = pd.DataFrame({
    "ID": [4, 5, 6],
    "Nome": ["David", "Eva", "Frank"],
    "Età": ["40", "45", "50"],  # intenzionalmente come stringhe
})

In [ ]:
# salviamo i DataFrame come file CSV
df1.to_csv("dati1.csv", index=False)
df2.to_csv("dati2.csv", index=False)
print("File 'dati1.csv' e 'dati2.csv' creati.")

In [ ]:
from glob import glob

# cerchiamo i file CSV che iniziano con "dati"
file_csv = sorted(glob("dati*.csv"))
print("File CSV trovati:")
for file in file_csv:
    print(file)

In [ ]:
# leggere il primo file CSV
df_csv1 = pd.read_csv("dati1.csv")
df_csv1

<a id="sez-4-2"></a>
### 4.2 Specificare i tipi di dato

Durante la lettura pandas prova a riconoscere da solo il tipo di ogni colonna. A volte serve
indicarlo noi, con il parametro `dtype`. Supponiamo di volere la colonna `Età` sicuramente come intero.

In [ ]:
# leggere il CSV specificando i tipi di dato
tipi_colonne = {"ID": int, "Nome": str, "Età": int}
df_csv1_tipizzato = pd.read_csv("dati1.csv", dtype=tipi_colonne)
df_csv1_tipizzato.dtypes

Altri parametri utili di `read_csv()`:

- **`usecols`**: quali colonne leggere dal file;
- **`sep`**: il delimitatore (`,`, `;`, `\t`);
- **`decimal`**: il separatore dei decimali (per esempio `,` invece di `.`).

Per unire i dati di più file leggiamo ogni file in un DataFrame, mettiamo i DataFrame in una lista e
li concateniamo con `pd.concat()`.

In [ ]:
# leggere tutti i file CSV e aggiungerli a una lista
dfs = []
for nome_file in file_csv:
    df = pd.read_csv(nome_file, dtype=tipi_colonne)
    dfs.append(df)

In [ ]:
# concatenare i DataFrame
df_unito = pd.concat(dfs, ignore_index=True)
df_unito

<a id="sez-4-3"></a>
### 4.3 Un CSV salvato da Excel in italiano

`letture_pod_2025.csv` contiene i consumi mensili per fascia di sei POD, esportati da un Excel
italiano. Questa cella dà errore apposta: leggiamo l'ultima riga del messaggio.

In [ ]:
pd.read_csv("../Dati/letture_pod_2025.csv")

`UnicodeDecodeError`: pandas legge il file come UTF-8 e trova un byte che non lo è (la `è` di
"Caffè"). I file salvati da Excel su Windows usano spesso l'encoding `latin-1`. In più i CSV italiani
separano le colonne con `;` e scrivono i decimali con la virgola: servono tre parametri.

In [ ]:
letture = pd.read_csv("../Dati/letture_pod_2025.csv", sep=";", decimal=",", encoding="latin-1")
letture.head()

In [ ]:
letture.dtypes

`kwh` è `float64`: senza `decimal=","` sarebbe rimasta testo, e una colonna di testo non si somma.
`encoding="latin-1"` si aggiunge solo dopo aver visto l'errore: su un file UTF-8 con lettere
accentate produrrebbe caratteri strani senza segnalarlo.

<a id="sez-5"></a>
## 5. File Excel

I file Excel sono molto usati per i dati strutturati e possono contenere più fogli. Il carico
elettrico della zona Nord pubblicato da Terna è in un file Excel.

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico.head()

<a id="sez-5-1"></a>
### 5.1 Più fogli in un file

`pd.ExcelWriter` scrive più DataFrame nello stesso file, ognuno in un foglio. Con `read_excel()`
scegliamo il foglio da leggere con `sheet_name`.

In [ ]:
# creare un file Excel con più fogli
with pd.ExcelWriter("dati.xlsx") as writer:
    df1.to_excel(writer, sheet_name="Foglio1", index=False)
    df2.to_excel(writer, sheet_name="Foglio2", index=False)
print("File 'dati.xlsx' creato.")

In [ ]:
# leggere un foglio specifico
df_foglio1 = pd.read_excel("dati.xlsx", sheet_name="Foglio1", dtype=tipi_colonne)
df_foglio1.dtypes

Con `sheet_name=None` pandas legge tutti i fogli e restituisce un dizionario: la chiave è il nome del
foglio, il valore il suo DataFrame. Poi li uniamo con `pd.concat()`.

In [ ]:
# leggere tutti i fogli dal file Excel
fogli = pd.read_excel("dati.xlsx", sheet_name=None, dtype=tipi_colonne)
fogli.keys()

In [ ]:
# concatenare i DataFrame
df_excel_unito = pd.concat(fogli.values(), ignore_index=True)
df_excel_unito

<a id="sez-6"></a>
## 6. Database SQL

Spesso i dati stanno in un database: tabelle interrogate con SQL. Python ci parla attraverso una
connessione, e `pd.read_sql` restituisce il risultato di una query come DataFrame. Qui usiamo SQLite,
un database contenuto in un singolo file, `utility.db`.

In [ ]:
import sqlite3

con = sqlite3.connect("../Dati/utility.db")
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)

`sqlite_master` è la tabella in cui SQLite tiene l'elenco del suo contenuto: le tabelle sono
`clienti`, `pod` e `letture`. `SELECT * FROM letture` vuol dire: tutte le colonne e tutte le righe.

In [ ]:
letture_db = pd.read_sql("SELECT * FROM letture", con)
letture_db.head()

Con `WHERE` il filtro lo fa il database, e arrivano solo le righe che servono.

In [ ]:
pd.read_sql("SELECT * FROM pod WHERE potenza_kw >= 30", con)

Alla fine si chiude la connessione.

In [ ]:
con.close()

<a id="sez-7"></a>
## 7. Lettura di dati da API

Le API (*Application Programming Interface*) permettono a programmi diversi di comunicare tra
loro. Molti servizi web offrono API pubbliche per accedere ai loro dati: si invia una richiesta
HTTP (per esempio una GET) a un indirizzo, l'endpoint, e si riceve una risposta, spesso in JSON.

<a id="sez-7-1"></a>
### 7.1 JSON e requests

JSON (*JavaScript Object Notation*) è un formato di testo per lo scambio di dati, fatto di coppie
chiave-valore simili ai dizionari Python.

```json
{
  "name": "Alice",
  "age": 30,
  "city": "Rome",
  "interests": ["reading", "hiking", "coding"]
}
```

La libreria `requests` invia le richieste HTTP. I passi sono sempre gli stessi: definire
l'endpoint e i parametri, inviare la richiesta, convertire il JSON ricevuto in un DataFrame.

Usiamo gli open data di Regione Lombardia sulle stazioni meteo:

- anagrafica dei sensori: `https://www.dati.lombardia.it/resource/nf78-nj6b.json`;
- misure dei sensori: `https://www.dati.lombardia.it/resource/647i-nhxk.json`.

In [ ]:
import requests

url = "https://www.dati.lombardia.it/resource/nf78-nj6b.json"
response = requests.get(url, timeout=30)
response

Ogni risposta ha un *codice di stato* che dice com'è andata la richiesta:

- **200**: la richiesta è stata completata correttamente;
- **404**: risorsa non trovata, l'URL potrebbe essere sbagliato;
- **500**: errore del server.

In [ ]:
risposta_json = response.json()
len(risposta_json)

In [ ]:
risposta_json[0]

<a id="sez-7-2"></a>
### 7.2 Parametri di query

Spesso non vogliamo tutti i dati, ma solo una parte. I **parametri di query** si aggiungono alla
fine dell'URL: iniziano dopo il simbolo `?`, sono coppie `chiave=valore` e si separano con `&`.

In [ ]:
# URL con i parametri di query per la provincia di Milano
url = "https://www.dati.lombardia.it/resource/nf78-nj6b.json?provincia=MI&tipologia=Precipitazione"
response = requests.get(url, timeout=30)
dati = response.json()
dati[:4]

Con `requests` i parametri si passano in un dizionario con l'argomento `params`: il codice è più
leggibile e `requests` costruisce l'URL per noi.

In [ ]:
parametri = {
    "provincia": "MI",
    "tipologia": "Precipitazione",
}
url = "https://www.dati.lombardia.it/resource/nf78-nj6b.json"
response = requests.get(url, params=parametri, timeout=30)
dati = response.json()
dati[:4]

<a id="sez-7-3"></a>
### 7.3 L'anagrafica dei sensori

Recuperiamo le informazioni su tutti i sensori. I parametri speciali iniziano con `$`: `$limit`
dice quante righe vogliamo (senza, il portale ne restituisce mille). `raise_for_status()` ferma la
cella se il codice di stato è un errore.

In [ ]:
url_sensori = "https://www.dati.lombardia.it/resource/nf78-nj6b.json"
parametri = {"$limit": 5000}
response = requests.get(url_sensori, params=parametri, timeout=30)
response.raise_for_status()

sensori_df = pd.DataFrame(response.json())
sensori_df.head()

Se l'API non risponde, esegui la cella qui sotto: legge dati di esempio da un file di riserva.

In [ ]:
import json

with open("../Dati/fallback/lombardia_sensori.json", encoding="utf-8") as f:
    sensori_df = pd.DataFrame(json.load(f))
sensori_df.head()

Quali tipi di sensori ci sono, e in quali province?

In [ ]:
sensori_df["tipologia"].unique()

In [ ]:
sensori_df["provincia"].unique()

Il filtro con due condizioni lo vediamo nel notebook sulle operazioni: qui basta leggerlo.

Teniamo i sensori di temperatura della provincia di Milano: tra parentesi quadre mettiamo le due
condizioni, unite da `&`.

In [ ]:
sensori_milano = sensori_df[(sensori_df["tipologia"] == "Temperatura") & (sensori_df["provincia"] == "MI")]
sensori_milano.head()

<a id="sez-7-4"></a>
### 7.4 Le misure di un sensore

Con l'ID di un sensore recuperiamo le sue misure dall'altra risorsa. Scegliamo il sensore 2001, il
termometro di Milano via Brera.

In [ ]:
idsensore = "2001"

In [ ]:
url_misure = "https://www.dati.lombardia.it/resource/647i-nhxk.json"
parametri_misure = {
    "idsensore": idsensore,
    "$limit": 100000,
}
response = requests.get(url_misure, params=parametri_misure, timeout=30)
response.raise_for_status()

misure_df = pd.DataFrame(response.json())
misure_df

Se l'API non risponde, esegui la cella qui sotto: legge dati di esempio da un file di riserva.

In [ ]:
with open("../Dati/fallback/lombardia_misure_2001.json", encoding="utf-8") as f:
    misure_df = pd.DataFrame(json.load(f))
misure_df

Le conversioni e il grafico qui sotto anticipano i notebook sulle date e su Plotly.

Il portale manda tutto come testo: convertiamo la data in datetime e il valore in numero, poi
ordiniamo per data. Il valore `-9999` indica una misura mancante: togliamo quelle righe.

In [ ]:
misure_df["data"] = pd.to_datetime(misure_df["data"])
misure_df["valore"] = pd.to_numeric(misure_df["valore"], errors="coerce")
misure_df = misure_df.sort_values("data")
misure_df = misure_df[misure_df["valore"] != -9999]

Visualizziamo l'andamento della temperatura nelle ultime 100 misure.

In [ ]:
import plotly.express as px

df_grafico = misure_df.iloc[-100:]

fig = px.line(
    df_grafico,
    x="data",
    y="valore",
    markers=True,
    title=f"Andamento della temperatura per IDsensore {idsensore}",
    labels={"data": "Data e ora", "valore": "Temperatura (°C)"},
)
fig.show()

Riassunto in una pagina: [Da Excel a pandas](../Schede/Scheda_Excel_pandas.md).

<a id="sez-8"></a>
## 8. Esercizi

<a id="es-6-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 6.1 · Lettura di un CSV italiano**

Delle letture dei POD ci servono solo tre colonne e il consumo complessivo.

**Cosa fare.**

1. Leggi `../Dati/letture_pod_2025.csv` nel DataFrame `letture_kwh`, tenendo solo le colonne
   `pod`, `fascia` e `kwh`.
2. Metti in `totale_kwh` la somma della colonna `kwh`.

*Output atteso:* 216 righe, 3 colonne, un totale di circa 134507.7 kWh.

*Suggerimento:* i parametri sono `sep`, `decimal`, `encoding` e `usecols`; la somma di una colonna è `.sum()`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 6.1**

Senza `decimal=","` la colonna `kwh` resterebbe testo e la somma non darebbe un numero.

</div>

In [ ]:
letture_kwh = pd.read_csv(
    "../Dati/letture_pod_2025.csv",
    sep=";",
    decimal=",",
    encoding="latin-1",
    usecols=["pod", "fascia", "kwh"],
)
totale_kwh = letture_kwh["kwh"].sum()
totale_kwh

In [ ]:
from corso import verifica
verifica("6.1")

<a id="es-6-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 6.2 · Due fogli Excel in una tabella**

Un negozio tiene le vendite di gennaio e di febbraio in due fogli dello stesso file Excel.

**Cosa fare.**

1. Scrivi `gennaio` e `febbraio` nel file `vendite.xlsx`, nei fogli `Gennaio` e `Febbraio`.
2. Leggi tutti i fogli del file nel dizionario `fogli`.
3. Concatenali nel DataFrame `vendite`, con un indice da 0 a 5.

*Output atteso:* 6 righe con le colonne `prodotto` e `pezzi`.

*Suggerimento:* `pd.ExcelWriter`, poi `sheet_name=None` e `pd.concat(..., ignore_index=True)`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 6.2**

</div>

In [ ]:
gennaio = pd.DataFrame({"prodotto": ["pane", "latte", "caffè"], "pezzi": [120, 80, 45]})
febbraio = pd.DataFrame({"prodotto": ["pane", "latte", "caffè"], "pezzi": [110, 95, 50]})

with pd.ExcelWriter("vendite.xlsx") as writer:
    gennaio.to_excel(writer, sheet_name="Gennaio", index=False)
    febbraio.to_excel(writer, sheet_name="Febbraio", index=False)

fogli = pd.read_excel("vendite.xlsx", sheet_name=None)
vendite = pd.concat(fogli.values(), ignore_index=True)
vendite

In [ ]:
from corso import verifica
verifica("6.2")

<a id="es-6-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 6.3 · Le letture di un POD dal database (facoltativo)**

Un POD è il codice che identifica un punto di prelievo. Dal database `utility.db` ci servono le letture di marzo di un solo POD.

**Cosa fare.**

1. Apri la connessione a `../Dati/utility.db`.
2. Con una query, leggi dalla tabella `letture` solo le righe del POD `IT001E10000000` nel
   DataFrame `letture_pod`.
3. Metti in `totale_marzo` la somma della colonna `kwh`, poi chiudi la connessione.

*Output atteso:* 31 righe, una per giorno, e un totale di 7785.6 kWh.

*Suggerimento:* dentro la query il testo va tra apici singoli: `WHERE pod = '...'`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 6.3**

</div>

In [ ]:
con = sqlite3.connect("../Dati/utility.db")
letture_pod = pd.read_sql("SELECT * FROM letture WHERE pod = 'IT001E10000000'", con)
totale_marzo = letture_pod["kwh"].sum()
con.close()
totale_marzo

In [ ]:
from corso import verifica
verifica("6.3")

<a id="es-6-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 6.4 · La certificazione energetica degli edifici in Lombardia (facoltativo)**

Regione Lombardia pubblica i dati sulla certificazione energetica degli edifici. Questo
esercizio richiede la rete: non c'è un file di riserva.

**Cosa fare.**

1. Recupera i dati dall'endpoint `https://www.dati.lombardia.it/resource/rsg3-xhvk.json` nel
   DataFrame `certificati`.
2. Tieni solo gli edifici della provincia di Milano nel DataFrame `milano`.
3. Calcola le emissioni medie di CO2 per ciascuna classe energetica in `co2_per_classe`.

*Output atteso:* una Series con una riga per classe energetica (A4, A3, ..., G) e le emissioni
medie di CO2.

*Suggerimento:* guarda prima `certificati.columns` e i valori con `unique()`; i numeri arrivano come testo
(`pd.to_numeric`); la media per gruppo si scrive `df.groupby("colonna")["altra"].mean()`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 6.4**

I nomi delle colonne e i valori della provincia vanno controllati con `certificati.columns` e
`unique()`: se il portale li cambia, si adattano la maschera e il `groupby`.

</div>

In [ ]:
url = "https://www.dati.lombardia.it/resource/rsg3-xhvk.json"
response = requests.get(url, params={"$limit": 50000}, timeout=30)
response.raise_for_status()
certificati = pd.DataFrame(response.json())

milano = certificati[certificati["provincia"] == "MILANO"].copy()
milano["emissioni_di_co2"] = pd.to_numeric(milano["emissioni_di_co2"], errors="coerce")
co2_per_classe = milano.groupby("classe_energetica")["emissioni_di_co2"].mean()
co2_per_classe

In [ ]:
from corso import verifica
verifica("6.4")

---

**Fine del notebook 06.** Precedente: [05 · Oggetti ed errori](05_Oggetti_ed_errori.ipynb) &nbsp;·&nbsp; Prossimo: [E2 · Esercitazione 2](Esercitazione_2.ipynb)

<sub>Blocco 2 · Controllo del flusso, funzioni e primi dati in pandas · Giornata 1 · Aula Avanzata</sub>